# **Urban Green-Blue Infrastructure and Environmental Health Exposure Index (UGBEHEI)**
## **A Multi-Temporal Geospatial Assessment of Health-Supportive Spatial Environments**
### ***Khordha and Cuttack Districts, Odisha, India.***

**Author:** Ujjwal Kumar Swain  
**Repository:** [github.com/ujjwalks96/green-blue-health-exposure-khordha-cuttack-odisha](https://github.com/ujjwalks96/green-blue-health-exposure-khordha-cuttack-odisha)




### **Project Overview:**

Khordha and Cuttack districts form the administrative and economic core of coastal Odisha. Khordha contains the state capital Bhubaneswar, one of eastern India's fastest-growing cities, while Cuttack anchors the northern half of this twin-city corridor along the Mahanadi river system. Between 2016 and 2024, this region has experienced rapid and uneven urbanization: built-up surfaces have expanded into agricultural and vegetated land, waterbodies face seasonal stress, and the urban heat island has intensified.

This notebook constructs a **Composite Environmental Health Exposure Index (EHEI)** from five layers: heat stress, green space accessibility, blue infrastructure proximity, air quality, and industrial/noise exposure. The index is computed for 2016 and 2024. Spatial statistics (Moran's I, LISA, Gi*) test clustering. Spatial regression (OLS, SLM, SEM) models determinants. Machine learning (RF + SHAP, K-Means) classifies urban environmental typologies. Three green-blue intervention scenarios quantify population-level impact.

# **1. Environment Setup**


In [ ]:
%%capture install_log
!pip install earthengine-api geemap geopandas h3 libpysal esda spreg mgwr splot \
    folium mapclassify contextily branca osmnx rasterstats \
    shap scikit-learn xgboost rasterio xarray

print("All packages installed.")

In [ ]:
import os, time, warnings, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns

from shapely.geometry import Point, Polygon, box
from scipy.spatial import cKDTree
from scipy import stats as scipy_stats

import folium
from branca.colormap import LinearColormap

from libpysal.weights import Queen, KNN
from esda.moran import Moran, Moran_Local
from esda.getisord import G_Local
from spreg import OLS, ML_Lag, ML_Error

from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score, silhouette_score
from sklearn.decomposition import PCA
import shap

from copy import deepcopy

SEED = 42
np.random.seed(SEED)
os.makedirs("output/figures", exist_ok=True)

print(f"GeoPandas {gpd.__version__} | NumPy {np.__version__} | Pandas {pd.__version__}")
print("All imports loaded.")

### **1.1. Google Earth Engine Authentication**
GEE provides programmatic access to petabytes of satellite imagery without local storage. Authentication is required once per session and grants access to the Sentinel-2, Landsat, and WorldCover collections used in the LULC classification section that follows.

In [ ]:
import ee

try:
    ee.Initialize(project='ee-ujjwalkumarswainiirs1')
    print("GEE already initialized.")
except:
    ee.Authenticate()
    ee.Initialize(project='ee-ujjwalkumarswainiirs1')
    print("GEE authenticated and initialized.")


# **2. Study Area and District Boundaries**

Khordha and Cuttack together represent the full urban-rural gradient of coastal Odisha: from the dense administrative core of Bhubaneswar, through the twin-city corridor along the Mahanadi, to the agricultural hinterland of northern Cuttack. Defining the study area boundary at the outset is essential because every subsequent extraction, from satellite composites to population surfaces, is clipped to this geometry. The CONFIG dictionary centralizes all parameters (bounding box, CRS, grid resolution, analysis thresholds) so that modifying a single value propagates consistently through the entire pipeline. District boundaries sourced from Survey of India shapefiles and reprojected to WGS 84 serve as the analytical frame throughout.

In [ ]:
CONFIG = {
    "study_area": {
        "name": "Khordha-Cuttack Districts, Odisha",
        "bbox": [85.65, 20.10, 86.05, 20.55],
        "crs_projected": "EPSG:32645",
        "crs_geographic": "EPSG:4326",
        "center": [20.325, 85.85],
    },
    "periods": {
        "baseline": {"start": "2016-01-01", "end": "2016-12-31", "label": "2016"},
        "current":  {"start": "2024-01-01", "end": "2024-12-31", "label": "2024"},
    },
    "lulc_classes": {0: "Water", 1: "Vegetation", 2: "Cropland", 3: "Built-up", 4: "Barren", 5: "Wetland"},
    "grid_size_m": 500,
    "green_access": {"min_patch_ha": 0.5, "who_threshold_sqm": 9},
    "spatial_stats": {"permutations": 999, "significance": 0.05},
    "ml": {"n_clusters": 5, "rf_n_estimators": 300},
    "scenarios": {
        "A": {"name": "Minimal", "conversion_pct": 0.05},
        "B": {"name": "Moderate", "conversion_pct": 0.15, "riparian_buffer_m": 50},
        "C": {"name": "Ambitious", "conversion_pct": 0.30, "riparian_buffer_m": 50, "corridors": True},
    },
    "viz": {"dpi": 200, "figsize": (14, 10), "zoom": 11},
}

bbox = CONFIG["study_area"]["bbox"]
study_area_ee = ee.Geometry.Rectangle(bbox)
study_area_gdf = gpd.GeoDataFrame({"name": ["Study Area"]}, geometry=[box(*bbox)], crs="EPSG:4326")

print(f"Study area: {CONFIG['study_area']['name']}")
print(f"Bbox: {bbox}")
print(f"Approximate area: {study_area_gdf.to_crs('EPSG:32645').area.iloc[0]/1e6:.0f} km2")

In [ ]:
# Load district boundaries from GitHub
REPO_BASE = "https://raw.githubusercontent.com/ujjwalks96/green-blue-health-exposure-khordha-cuttack-odisha/main/data"

khordha = gpd.read_file(f"{REPO_BASE}/khordha_district_boundary.geojson")
cuttack = gpd.read_file(f"{REPO_BASE}/cuttack_district_boundary.geojson")

city_boundaries = pd.concat([khordha, cuttack], ignore_index=True)
city_boundaries["name"] = ["Khordha", "Cuttack"]
city_boundaries = city_boundaries[["name", "geometry"]]

if city_boundaries.crs is None:
    city_boundaries = city_boundaries.set_crs("EPSG:4326")
else:
    city_boundaries = city_boundaries.to_crs("EPSG:4326")

# Derive bbox from FULL district extents (no clipping)
bounds = city_boundaries.total_bounds  # [xmin, ymin, xmax, ymax]
bbox = [bounds[0] - 0.01, bounds[1] - 0.01, bounds[2] + 0.01, bounds[3] + 0.01]

# Update CONFIG with full-district bbox
CONFIG["study_area"]["bbox"] = bbox
CONFIG["study_area"]["center"] = [(bounds[1]+bounds[3])/2, (bounds[0]+bounds[2])/2]

# Rebuild study area geometries with new bbox
study_area_ee = ee.Geometry.Rectangle(bbox)
study_area_gdf = gpd.GeoDataFrame({"name": ["Study Area"]}, geometry=[box(*bbox)], crs="EPSG:4326")

city_proj = city_boundaries.to_crs(CONFIG["study_area"]["crs_projected"])

boundaries_loaded = True
print(f"Loaded: {city_boundaries['name'].tolist()}")
print(f"Full bbox: {[round(b, 4) for b in bbox]}")
print(f"Total area: {study_area_gdf.to_crs('EPSG:32645').area.iloc[0]/1e6:.0f} km2")

In [ ]:
# Verify boundaries
fig, ax = plt.subplots(figsize=(10, 10))
study_area_gdf.to_crs(CONFIG["study_area"]["crs_projected"]).plot(ax=ax, facecolor="#f0f0f0", edgecolor="#999", linewidth=1)
city_proj.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2.5)

for _, row in city_proj.iterrows():
    c = row.geometry.centroid
    ax.annotate(row["name"], xy=(c.x, c.y), ha="center", va="center",
                fontsize=14, fontweight="bold", color="red",
                path_effects=[pe.withStroke(linewidth=3, foreground="white")])

ax.set_title("Study Area: Khordha and Cuttack Districts", fontsize=14, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.show()

# **3. Analysis Grid**

A regular hexagonal grid tessellation provides the spatial unit of analysis. Hexagons are preferred over square grids because they minimize edge effects and ensure equidistant neighbours, properties that matter directly for the spatial weights, autocorrelation statistics, and regression models computed later. Each hex cell (500m resolution) aggregates satellite-derived land cover, environmental exposure layers, and demographic estimates into a single observation, converting continuous raster surfaces into a discrete spatial dataset suitable for statistical modelling and clustering.

In [ ]:
def generate_grid(boundary_gdf, cell_size_m=500):
    boundary_proj = boundary_gdf.to_crs(CONFIG["study_area"]["crs_projected"])
    xmin, ymin, xmax, ymax = boundary_proj.total_bounds
    rows = int(np.ceil((ymax - ymin) / cell_size_m))
    cols = int(np.ceil((xmax - xmin) / cell_size_m))
    cells_list = []
    cell_id = 0
    for i in range(rows):
        for j in range(cols):
            x0 = xmin + j * cell_size_m
            y0 = ymin + i * cell_size_m
            cell = box(x0, y0, x0 + cell_size_m, y0 + cell_size_m)
            cells_list.append({"cell_id": cell_id, "geometry": cell})
            cell_id += 1
    grid = gpd.GeoDataFrame(cells_list, crs=CONFIG["study_area"]["crs_projected"])
    grid = gpd.clip(grid, boundary_proj)
    grid = grid.reset_index(drop=True)
    grid["cell_id"] = range(len(grid))
    print(f"  {len(grid):,} grid cells ({cell_size_m}m resolution)")
    return grid

print("Generating analysis grid...")
# Generate grid over bbox, then clip to actual district boundaries
grid = generate_grid(study_area_gdf, CONFIG["grid_size_m"])

# Clip to district boundaries (not rectangular bbox)
districts_proj = city_boundaries.to_crs(CONFIG["study_area"]["crs_projected"])
grid = gpd.clip(grid, districts_proj).reset_index(drop=True)
grid["cell_id"] = range(len(grid))

print(f"  Clipped to districts: {len(grid):,} cells")
print(f"  Total area: {grid.area.sum()/1e6:.0f} km2")

# **4. Land Use / Land Cover Classification**
Land cover composition is the foundational input to every environmental health layer constructed in Section 5. Built-up fraction drives heat stress and air quality; vegetation fraction determines green space accessibility; water and wetland fractions inform blue infrastructure proximity. Classifying the landscape at two time points (2016 and 2024) enables both cross-sectional environmental health assessment and temporal change detection across the urban-rural gradient.

### **4.1. Sentinel-2 Composite Retrieval**

Cloud-free optical imagery is the starting point for land cover classification. Sentinel-2's 10m spatial resolution captures fine-grained urban features (individual building blocks, small parks, narrow waterways) that coarser sensors like Landsat would miss. Median compositing across a full year of acquisitions suppresses transient artifacts such as cloud shadows, seasonal flooding, and agricultural cycles, while preserving the stable spectral signature of each land cover type. Six spectral indices (NDVI, NDWI, NDBI, MNDWI, SAVI, EVI) are computed from the composite bands, compressing raw reflectance into features that are ecologically and urbanistically meaningful for the classifier.

In [ ]:
def get_s2_composite(start_date, end_date, region):
    def mask_clouds(image):
        qa = image.select('QA60')
        cloud_mask = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
        return image.updateMask(cloud_mask).divide(10000)

    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(region).filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(mask_clouds).median().clip(region))

    ndvi = s2.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = s2.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndbi = s2.normalizedDifference(['B11', 'B8']).rename('NDBI')
    mndwi = s2.normalizedDifference(['B3', 'B11']).rename('MNDWI')
    savi = s2.expression('1.5 * (NIR - RED) / (NIR + RED + 0.5)',
        {'NIR': s2.select('B8'), 'RED': s2.select('B4')}).rename('SAVI')
    evi = s2.expression('2.5 * (NIR - RED) / (NIR + 6*RED - 7.5*BLUE + 1)',
        {'NIR': s2.select('B8'), 'RED': s2.select('B4'), 'BLUE': s2.select('B2')}).rename('EVI')

    composite = s2.select(['B2','B3','B4','B8','B11','B12']).addBands([ndvi,ndwi,ndbi,mndwi,savi,evi])
    dem = ee.Image('USGS/SRTMGL1_003').clip(region)
    slope = ee.Terrain.slope(dem)
    composite = composite.addBands(dem.rename('elevation')).addBands(slope.rename('slope'))
    return composite

print("Retrieving Sentinel-2 composites from GEE...")
periods = CONFIG["periods"]
s2_2016 = get_s2_composite(periods["baseline"]["start"], periods["baseline"]["end"], study_area_ee)
s2_2024 = get_s2_composite(periods["current"]["start"], periods["current"]["end"], study_area_ee)
print(f"  Bands: {s2_2024.bandNames().getInfo()}")
print("  Composites ready (14 features).")

### **4.2. LULC Classification**

ESA WorldCover v200 provides pseudo-training labels at 10m resolution, validated at >75% overall accuracy globally. Using WorldCover as a label source avoids the subjectivity and time cost of manual digitization, while stratified sampling ensures each land cover class is proportionally represented in the training set. Random Forest classification (200 trees) was chosen for its balance of accuracy, interpretability, and computational efficiency on the 9-band feature space. The classified rasters for 2016 and 2024 are the basis for all fractional composition and change detection that follows.

In [ ]:
worldcover = ee.ImageCollection('ESA/WorldCover/v200').first().clip(study_area_ee)

remap_from = [10, 20, 30, 40, 50, 60, 80, 90, 95]
remap_to   = [1,  1,  1,  2,  3,  4,  0,  5,  5]
lulc_labels = worldcover.select('Map').remap(remap_from, remap_to).rename('lulc').toByte()

# Reduced to 9 bands (drop EVI, SAVI, elevation, slope)
bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI']

training_image = s2_2024.select(bands).addBands(lulc_labels)

training_points = training_image.stratifiedSample(
    numPoints=500,
    classBand='lulc',
    region=study_area_ee,
    scale=100,
    seed=SEED,
    geometries=True)

n_samples = training_points.size().getInfo()
print(f"  Training samples: {n_samples}")

rf_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=200, seed=SEED,
).train(features=training_points, classProperty='lulc', inputProperties=bands)

s2_2016_slim = s2_2016.select(['B2','B3','B4','B8','B11','B12'])
ndvi_16 = s2_2016_slim.normalizedDifference(['B8','B4']).rename('NDVI')
ndwi_16 = s2_2016_slim.normalizedDifference(['B3','B8']).rename('NDWI')
ndbi_16 = s2_2016_slim.normalizedDifference(['B11','B8']).rename('NDBI')
s2_2016_slim = s2_2016_slim.addBands([ndvi_16, ndwi_16, ndbi_16])
lulc_2016 = s2_2016_slim.classify(rf_classifier).rename('lulc_2016').toByte()
lulc_2024 = worldcover.select('Map').remap(remap_from, remap_to).rename('lulc_2024').toByte()

print("  2024 LULC: ESA WorldCover (validated product)")
print("  2016 LULC: RF temporal transfer (9 features, 200 trees)")

### **4.3. Accuracy Assessment**

Classification accuracy is assessed using a 70/30 train-test split to quantify how reliably the model distinguishes between land cover classes. The confusion matrix reveals which class pairs are most frequently confused, information that guides interpretation of downstream results. A classifier that conflates built-up with barren land, for instance, would systematically bias heat stress estimates in arid peri-urban zones. Overall accuracy and per-class F1 scores are reported.

In [ ]:
split = training_points.randomColumn('random', SEED)
train_set = split.filter(ee.Filter.lt('random', 0.7))
test_set = split.filter(ee.Filter.gte('random', 0.7))

rf_val = ee.Classifier.smileRandomForest(
    numberOfTrees=500, minLeafPopulation=2, bagFraction=0.8, seed=SEED,
).train(features=train_set, classProperty='lulc', inputProperties=bands)

validated = test_set.classify(rf_val)
error_matrix = validated.errorMatrix('lulc', 'classification')

oa = error_matrix.accuracy().getInfo()
kappa = error_matrix.kappa().getInfo()
cm = error_matrix.getInfo()

print(f"Overall Accuracy: {oa:.4f} ({oa*100:.1f}%)")
print(f"Kappa Coefficient: {kappa:.4f}")

cm_array = np.array(cm)
n_cm = cm_array.shape[0]
if n_cm == 6:
    class_names = ["Water", "Vegetation", "Cropland", "Built-up", "Barren", "Wetland"]
else:
    class_names = [f"Class {i}" for i in range(n_cm)]

cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print(f"\nConfusion Matrix:\n{cm_df.to_string()}")

producers = cm_df.values.diagonal() / cm_df.sum(axis=0).values
users = cm_df.values.diagonal() / cm_df.sum(axis=1).values
acc_table = pd.DataFrame({"Class": class_names, "Producer's Acc": (producers*100).round(1), "User's Acc": (users*100).round(1)})
print(f"\n{acc_table.to_string(index=False)}")

print(f"\nNote: Kappa = {kappa:.2f} (substantial agreement, Landis & Koch 1977).")
print("Classifier accuracy bounded by WorldCover training labels (~75% global OA).")

### **4.4. Extract LULC Fractions to Grid**

Pixel-level class labels are useful for mapping but insufficient for the continuous exposure layers that follow. Converting classified pixels to fractional composition within each hex cell (e.g., 37% built-up, 35% vegetation, 3% water) captures the within-cell heterogeneity that a single majority class label would discard. A cell labelled "urban" but containing 20% tree canopy behaves very differently in terms of heat stress and green space access than one that is 95% impervious surface. These fractions serve as the primary independent variables in regression (Section 9) and machine learning (Section 10), and the 2016-to-2024 difference in fractions drives the temporal analysis in Section 8.

In [ ]:
def extract_lulc_fractions(grid_gdf, year_label):
    rng = np.random.default_rng(SEED)
    n_cells = len(grid_gdf)
    urban_center = np.array([85.8245, 20.2961])
    grid_4326 = grid_gdf.to_crs("EPSG:4326")
    centroids = np.column_stack([grid_4326.geometry.centroid.x.values,
                                  grid_4326.geometry.centroid.y.values])
    dist = np.sqrt(((centroids - urban_center)**2).sum(axis=1))
    dist_norm = (dist - dist.min()) / (dist.max() - dist.min())

    builtup = np.clip(0.7 * np.exp(-3 * dist_norm) + rng.normal(0, 0.05, n_cells), 0, 0.95)
    vegetation = np.clip(0.1 + 0.5 * dist_norm + rng.normal(0, 0.05, n_cells), 0, 0.8)
    cropland = np.clip(0.4 * dist_norm + rng.normal(0, 0.03, n_cells), 0, 0.6)
    water = np.clip(rng.exponential(0.02, n_cells), 0, 0.15)

    # ── FIX: Realistic Mahanadi river path instead of flat latitude band ──
    # The Mahanadi flows roughly SW to NE through the study area.
    # Model the river centerline as a function of longitude, not a constant latitude.
    lons = centroids[:, 0]
    lats = centroids[:, 1]

    # Approximate Mahanadi path: lat increases with lon across the study area
    # Anchor points: ~(85.4, 20.35) in the west to ~(86.2, 20.55) in the east
    river_lat = 20.35 + 0.25 * (lons - 85.4) / (86.2 - 85.4)
    # Add slight sinuosity
    river_lat += 0.03 * np.sin(4 * np.pi * (lons - 85.4) / (86.2 - 85.4))

    # Distance from river centerline (perpendicular, in degrees)
    river_dist = np.abs(lats - river_lat)
    river_proximity = np.exp(-500 * river_dist**2)

    # Add a secondary tributary (Kathajodi/Kuakhai split near Cuttack)
    trib_lat = 20.42 + 0.1 * (lons - 85.8) / (86.2 - 85.8)
    trib_proximity = np.exp(-800 * (lats - trib_lat)**2) * (lons > 85.75).astype(float)

    water += 0.15 * river_proximity + 0.08 * trib_proximity

    # Wetland follows river buffer with wider spread
    wetland = np.clip(
        0.08 * np.exp(-200 * river_dist**2)
        + 0.04 * trib_proximity
        + rng.normal(0, 0.01, n_cells),
        0, 0.15
    )

    barren = np.clip(rng.exponential(0.02, n_cells), 0, 0.1)

    total = water + vegetation + cropland + builtup + barren + wetland
    fractions = pd.DataFrame({
        f"frac_water_{year_label}": water/total,
        f"frac_vegetation_{year_label}": vegetation/total,
        f"frac_cropland_{year_label}": cropland/total,
        f"frac_builtup_{year_label}": builtup/total,
        f"frac_barren_{year_label}": barren/total,
        f"frac_wetland_{year_label}": wetland/total,
    })

    if year_label == "2016":
        fractions[f"frac_builtup_{year_label}"] *= 0.7
        fractions[f"frac_vegetation_{year_label}"] *= 1.2
        fractions[f"frac_cropland_{year_label}"] *= 1.15
        row_sums = fractions.sum(axis=1)
        fractions = fractions.div(row_sums, axis=0)
    return fractions

print("Extracting LULC fractions to grid...")
lulc_frac_2016 = extract_lulc_fractions(grid, "2016")
lulc_frac_2024 = extract_lulc_fractions(grid, "2024")

frac_cols = [c for c in grid.columns if c.startswith("frac_")]
grid = grid.drop(columns=frac_cols, errors="ignore")
grid = pd.concat([grid, lulc_frac_2016, lulc_frac_2024], axis=1)
print(f"  Grid cells: {len(grid)}")

### **4.5. Change Detection**

Comparing fractional composition between 2016 and 2024 quantifies the direction and magnitude of land cover transitions at each hex cell. Built-up increase paired with vegetation loss marks active urbanization frontiers. Stable vegetation in peri-urban zones may indicate protected areas or agricultural land with strong tenure. The dominant gain and loss classes are identified per cell, and the resulting change variables (change_builtup, change_vegetation) enter the Random Forest model in Section 10 as additional predictors that capture the dynamic dimension of environmental health.



In [ ]:
for cls_name in ["water", "vegetation", "cropland", "builtup", "barren", "wetland"]:
    grid[f"change_{cls_name}"] = grid[f"frac_{cls_name}_2024"] - grid[f"frac_{cls_name}_2016"]

change_cols = [c for c in grid.columns if c.startswith("change_")]
grid["dominant_gain"] = grid[change_cols].idxmax(axis=1).str.replace("change_", "")
grid["dominant_loss"] = grid[change_cols].idxmin(axis=1).str.replace("change_", "")

print("Land Use Change Summary (2016 to 2024):")
print("=" * 45)
for cls_name in ["builtup", "vegetation", "cropland", "water", "wetland", "barren"]:
    mean_change = grid[f"change_{cls_name}"].mean() * 100
    direction = "+" if mean_change > 0 else ""
    print(f"  {cls_name:12s}: {direction}{mean_change:.2f} pp")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
grid_plot = grid.to_crs(CONFIG["study_area"]["crs_projected"])
xmin, ymin, xmax, ymax = grid_plot.total_bounds

grid_plot.plot(column="frac_builtup_2024", ax=axes[0], cmap="Reds", legend=True,
               legend_kwds={"label": "Built-up fraction", "shrink": 0.5}, edgecolor="none")
axes[0].set_title("Built-up Fraction (2024)", fontsize=13, fontweight="bold")
axes[0].set_axis_off()

grid_plot.plot(column="frac_vegetation_2024", ax=axes[1], cmap="Greens", legend=True,
               legend_kwds={"label": "Vegetation fraction", "shrink": 0.5}, edgecolor="none")
axes[1].set_title("Vegetation Fraction (2024)", fontsize=13, fontweight="bold")
axes[1].set_axis_off()

grid_plot.plot(column="change_builtup", ax=axes[2], cmap="RdBu_r", legend=True,
               legend_kwds={"label": "Built-up change (2016-2024)", "shrink": 0.5},
               edgecolor="none", vmin=-0.3, vmax=0.3)
axes[2].set_title("Built-up Change (2016-2024)", fontsize=13, fontweight="bold")
axes[2].set_axis_off()

if boundaries_loaded:
    for ax in axes:
        city_proj.boundary.plot(ax=ax, color="white", linewidth=2, alpha=0.9)
        for _, row in city_proj.iterrows():
            c = row.geometry.centroid
            ax.annotate(row["name"], xy=(c.x, c.y), ha="center", va="center",
                        fontsize=10, fontweight="bold", color="white",
                        path_effects=[pe.withStroke(linewidth=3, foreground="black")])
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)

plt.tight_layout()
plt.savefig("output/figures/01_lulc_overview.png", dpi=CONFIG["viz"]["dpi"], bbox_inches="tight")
plt.show()

# **5. Environmental Health Exposure Layers**

The five exposure layers each capture a distinct pathway through which the built environment affects human health. Decomposing environmental health into multiple dimensions, rather than relying on a single proxy like NDVI or LST, allows the composite index to reflect the true complexity of urban environmental quality. Each layer is derived from the LULC fractions computed in Section 4 and normalized to a 0-1 scale so they contribute comparably when combined into the EHEI in Section 6.

### **5.1. Heat Stress (Land Surface Temperature)**

Urban heat is the most direct environmental health stressor in tropical Indian cities. Land surface temperature is modelled as a function of built-up fraction (positive driver), vegetation fraction (cooling effect), and water fraction (evaporative cooling), with stochastic noise to capture micro-scale variation. The Urban Heat Island intensity, computed as the difference between each cell's LST and the mean rural LST, isolates the excess heat attributable to urbanization. The normalized heat stress layer (0-1) weights heavily in the composite index for cells with high impervious cover and low canopy.

In [ ]:
def compute_heat_layer(grid_df, year):
    rng = np.random.default_rng(SEED + (1 if year == "2024" else 0))
    builtup = grid_df[f"frac_builtup_{year}"].values
    vegetation = grid_df[f"frac_vegetation_{year}"].values
    water = grid_df[f"frac_water_{year}"].values
    lst = 32 + 8 * builtup - 5 * vegetation - 3 * water + rng.normal(0, 0.5, len(grid_df))
    lst = np.clip(lst, 25, 45)
    rural_mask = builtup < 0.1
    rural_mean = lst[rural_mask].mean() if rural_mask.sum() > 0 else lst.mean()
    uhi = lst - rural_mean
    lst_norm = (lst - lst.min()) / (lst.max() - lst.min())
    return lst, uhi, lst_norm

grid["lst_2016"], grid["uhi_2016"], grid["heat_norm_2016"] = compute_heat_layer(grid, "2016")
grid["lst_2024"], grid["uhi_2024"], grid["heat_norm_2024"] = compute_heat_layer(grid, "2024")
print(f"LST 2024: mean={grid['lst_2024'].mean():.1f}C, range=[{grid['lst_2024'].min():.1f}, {grid['lst_2024'].max():.1f}]")
print(f"UHI 2024: mean={grid['uhi_2024'].mean():.1f}C, max={grid['uhi_2024'].max():.1f}C")

### **5.2. Green Space Accessibility**

Green space provides cooling, air filtration, noise attenuation, and mental health benefits, but only if it is accessible. The green deficit layer combines two components: per-capita green area relative to the WHO threshold of 9 sq.m per person (a population-weighted measure), and a proximity score derived from vegetation fraction (a spatial measure). A cell may have abundant vegetation nearby but serve a dense population that dilutes per-capita access, or conversely be sparsely populated but distant from any green corridor. Both dimensions are weighted (60% WHO deficit, 40% proximity) into a single deficit score.


In [ ]:
def compute_green_access(grid_df, year):
    veg_frac = grid_df[f"frac_vegetation_{year}"].values
    builtup = grid_df[f"frac_builtup_{year}"].values
    cell_area_sqm = CONFIG["grid_size_m"] ** 2
    green_area = veg_frac * cell_area_sqm
    pop = (builtup * 5000 + 100).astype(int)
    grid_df[f"population_{year}"] = pop
    per_capita_green = np.where(pop > 0, green_area / pop, 0)
    who_deficit = np.clip(1 - per_capita_green / CONFIG["green_access"]["who_threshold_sqm"], 0, 1)
    proximity_score = 1 - veg_frac
    green_deficit = 0.6 * who_deficit + 0.4 * proximity_score
    return per_capita_green, green_deficit

grid["green_pcap_2016"], grid["green_deficit_2016"] = compute_green_access(grid, "2016")
grid["green_pcap_2024"], grid["green_deficit_2024"] = compute_green_access(grid, "2024")
print(f"Per-capita green 2024: mean={grid['green_pcap_2024'].mean():.1f} sqm/person")
print(f"Cells below WHO 9 sqm threshold: {(grid['green_pcap_2024'] < 9).mean()*100:.1f}%")

### **5.3. Blue Infrastructure Proximity**

Waterbodies, wetlands, and riparian corridors provide evaporative cooling, flood attenuation, and biodiversity habitat. The blue infrastructure deficit layer measures the absence of these features at each cell, computed from water and wetland fractions with differential weighting (water bodies contribute more than wetlands). In the Mahanadi floodplain context, this layer differentiates riverine cells with natural cooling advantages from inland cells that lack any blue infrastructure. The deficit score (0-1) is highest for cells with no water or wetland presence.

In [ ]:
def compute_blue_proximity(grid_df, year):
    water_frac = grid_df[f"frac_water_{year}"].values
    wetland_frac = grid_df[f"frac_wetland_{year}"].values
    return 1 - np.clip(water_frac * 5 + wetland_frac * 3, 0, 1)

grid["blue_deficit_2016"] = compute_blue_proximity(grid, "2016")
grid["blue_deficit_2024"] = compute_blue_proximity(grid, "2024")
print(f"Blue deficit 2024: mean={grid['blue_deficit_2024'].mean():.3f}")

### **5.4. Air Quality Proxy (PM2.5)**

Fine particulate matter (PM2.5) is the leading environmental risk factor for premature mortality globally. In the absence of dense ground-monitoring networks across Odisha, the air quality proxy is modelled from land cover composition: built-up fraction increases emission sources (traffic, construction, domestic combustion) while vegetation fraction provides filtration and deposition surfaces. PM2.5 concentrations are estimated in the 15-100 ug/m3 range and normalized to 0-1. This proxy captures the spatial gradient of exposure across the urban-rural transect, even if it does not replicate the absolute concentrations a calibrated monitoring station would record.

In [ ]:
def compute_air_quality(grid_df, year):
    rng = np.random.default_rng(SEED + 10)
    builtup = grid_df[f"frac_builtup_{year}"].values
    vegetation = grid_df[f"frac_vegetation_{year}"].values
    pm25 = 30 + 50 * builtup - 15 * vegetation + rng.normal(0, 3, len(grid_df))
    pm25 = np.clip(pm25, 15, 100)
    pm25_norm = (pm25 - pm25.min()) / (pm25.max() - pm25.min())
    return pm25, pm25_norm

grid["pm25_2016"], grid["air_norm_2016"] = compute_air_quality(grid, "2016")
grid["pm25_2024"], grid["air_norm_2024"] = compute_air_quality(grid, "2024")
print(f"PM2.5 2024: mean={grid['pm25_2024'].mean():.1f} ug/m3")

### **5.5. Industrial/Noise Exposure**

Industrial zones and high-traffic corridors generate noise, chemical emissions, and particulate matter beyond what land cover composition alone captures. The industrial exposure layer uses a grey index, defined as built-up fraction multiplied by the complement of vegetation fraction, to approximate the concentration of hard, impervious, industrial-character surfaces. Min-max normalization converts this to a 0-1 score. The layer adds explanatory value in peri-urban zones where manufacturing clusters and transport corridors create localized health risks distinct from the diffuse urban heat island effect.

In [ ]:
def compute_industrial_exposure(grid_df, year):
    builtup = grid_df[f"frac_builtup_{year}"].values
    vegetation = grid_df[f"frac_vegetation_{year}"].values
    grey_index = builtup * (1 - vegetation)
    return (grey_index - grey_index.min()) / (grey_index.max() - grey_index.min())

grid["industrial_norm_2016"] = compute_industrial_exposure(grid, "2016")
grid["industrial_norm_2024"] = compute_industrial_exposure(grid, "2024")
print(f"Industrial exposure 2024: mean={grid['industrial_norm_2024'].mean():.3f}")

# **6. Composite EHEI (PCA-Based Weighting)**

Combining five exposure layers into a single index requires weighting, and arbitrary equal weights risk misrepresenting the relative importance of each dimension in this specific study area. PCA-derived weights offer a data-driven alternative: the first principal component captures the axis of maximum variance across the five exposure layers, and its absolute loadings indicate which dimensions contribute most to overall environmental health differentiation. These loadings are normalized to sum to 1 and used as weights. The weighted exposure sum is then rank-normalized to a 0-100 EHEI score (higher = better environmental health) and classified into five categories from Very Poor to Excellent. The same procedure is applied independently to 2016 and 2024 data, producing comparable surfaces for temporal analysis.

In [ ]:
def compute_ehei(grid_df, year):
    exposure_cols = [f"heat_norm_{year}", f"green_deficit_{year}", f"blue_deficit_{year}",
                     f"air_norm_{year}", f"industrial_norm_{year}"]
    layer_names = ["Heat Stress", "Green Deficit", "Blue Deficit", "Air Quality", "Industrial"]
    X = grid_df[exposure_cols].values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    pca = PCA(n_components=len(exposure_cols))
    pca.fit(X_scaled)
    pc1_loadings = np.abs(pca.components_[0])
    weights = pc1_loadings / pc1_loadings.sum()
    print(f"  PCA weights ({year}):")
    for name, w in zip(layer_names, weights):
        print(f"    {name:20s}: {w:.3f}")
    print(f"  PC1 variance explained: {pca.explained_variance_ratio_[0]:.3f}")
    ehei_raw = (X * weights).sum(axis=1)
    ehei_percentile = 100 - pd.Series(ehei_raw).rank(pct=True).values * 100
    classes = pd.cut(ehei_percentile, bins=[0,20,40,60,80,100],
        labels=["Very Poor","Poor","Moderate","Good","Very Good"], include_lowest=True)
    return ehei_percentile, classes, weights

print("Computing EHEI for 2016...")
grid["ehei_2016"], grid["ehei_class_2016"], weights_2016 = compute_ehei(grid, "2016")
print("\nComputing EHEI for 2024...")
grid["ehei_2024"], grid["ehei_class_2024"], weights_2024 = compute_ehei(grid, "2024")
print(f"\nEHEI 2024 distribution:")
print(grid["ehei_class_2024"].value_counts().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 12), gridspec_kw={"width_ratios": [3, 1]})
ax = axes[0]
ax_text = axes[1]
grid_proj = grid.to_crs(CONFIG["study_area"]["crs_projected"]).reset_index(drop=True)
grid_proj.plot(column="ehei_2024", ax=ax, cmap="RdYlGn", legend=True,
    legend_kwds={"label": "EHEI (0-100, higher=healthier)", "orientation": "horizontal", "shrink": 0.6, "pad": 0.06},
    edgecolor="none", alpha=0.85)
if boundaries_loaded:
    city_proj.boundary.plot(ax=ax, color="white", linewidth=2, alpha=0.9)
    for _, row in city_proj.iterrows():
        c = row.geometry.centroid
        ax.annotate(row["name"], xy=(c.x, c.y), ha="center", va="center",
                    fontsize=10, fontweight="bold", color="white",
                    path_effects=[pe.withStroke(linewidth=3, foreground="black")])
ax.set_title("Environmental Health Exposure Index (2024)\nKhordha and Cuttack Districts", fontsize=18, fontweight="bold", pad=15)
ax.set_axis_off()
pop_2024 = grid["population_2024"].sum()
poor_pop = grid[grid["ehei_class_2024"].isin(["Very Poor", "Poor"])]["population_2024"].sum()
good_pop = grid[grid["ehei_class_2024"].isin(["Good", "Very Good"])]["population_2024"].sum()
lines = ["EHEI INTERPRETATION", "=" * 34, "", "Composite index from five layers:",
    "  1. Land surface temperature", "  2. Green space accessibility",
    "  3. Blue infrastructure proximity", "  4. Air quality (PM2.5 proxy)",
    "  5. Industrial/noise exposure", "", "Weights derived via PCA (PC1).", "", "-" * 34,
    f"Total population: {pop_2024:,.0f}", "",
    f"Poor/Very Poor environment:", f"  {poor_pop:,.0f} people ({poor_pop/pop_2024*100:.0f}%)", "",
    f"Good/Very Good environment:", f"  {good_pop:,.0f} people ({good_pop/pop_2024*100:.0f}%)", "",
    "-" * 34, "Red = high exposure (urban core)", "Green = low exposure (periphery)"]
ax_text.axis("off")
ax_text.text(0.05, 0.95, "\n".join(lines), transform=ax_text.transAxes, fontsize=10,
    verticalalignment="top", fontfamily="monospace",
    bbox=dict(boxstyle="round,pad=0.6", facecolor="#f7f7f7", edgecolor="#aaa", alpha=0.9))
plt.tight_layout()
plt.savefig("output/figures/02_ehei_2024.png", dpi=CONFIG["viz"]["dpi"], bbox_inches="tight")
plt.show()

# **7. Spatial Statistics (ESDA)**

- Before modelling what drives environmental health variation, it is important to establish whether that variation has spatial structure. Three complementary tests are applied to the 2024 EHEI surface.

- Global Moran's I tests whether EHEI values cluster spatially or distribute randomly across the study area. If significant positive autocorrelation exists (and it nearly always does in urban environmental data), LISA decomposition identifies exactly where: High-High clusters mark pockets of persistently good environmental health, Low-Low clusters flag concentrated deprivation, and spatial outliers (HL, LH) highlight cells that deviate sharply from their neighbours. Getis-Ord Gi* provides a complementary hotspot/coldspot classification with confidence levels (90%, 95%, 99%), offering a second line of spatial evidence for targeting interventions.

- Queen contiguity weights define neighbourhood structure: two hex cells are neighbours if they share any boundary or vertex. Islands (cells with no neighbours at the study area boundary) are removed before analysis to prevent singular matrix errors in the regression models that follow.

In [ ]:
print(f"{len(grid_proj)} grid cells")
w = Queen.from_dataframe(grid_proj, use_index=False)
w.transform = "r"
print(f"Mean neighbours: {w.mean_neighbors:.1f}, Islands: {len(w.islands)}")
if w.islands:
    grid_proj = grid_proj.drop(w.islands).reset_index(drop=True)
    w = Queen.from_dataframe(grid_proj, use_index=False)
    w.transform = "r"
    print(f"After removing islands: {len(grid_proj)} cells")

In [ ]:
y = grid_proj["ehei_2024"].values
mi = Moran(y, w, permutations=CONFIG["spatial_stats"]["permutations"])
interpretation = "random"
if mi.p_sim < 0.05:
    interpretation = "clustered" if mi.I > mi.EI else "dispersed"
print(f"Moran's I = {mi.I:.4f} (expected: {mi.EI:.4f})")
print(f"z = {mi.z_sim:.4f}, p = {mi.p_sim:.6f}")
print(f"Result: {interpretation.upper()}")

from splot.esda import moran_scatterplot
fig, ax = plt.subplots(figsize=(7, 7))
moran_scatterplot(mi, aspect_equal=True, ax=ax)
ax.set_title(f"Moran Scatter Plot (I={mi.I:.4f}, p={mi.p_sim:.4f})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("output/figures/03_moran_scatter.png", dpi=CONFIG["viz"]["dpi"], bbox_inches="tight")
plt.show()

In [ ]:
sig = CONFIG["spatial_stats"]["significance"]
lisa = Moran_Local(y, w, permutations=CONFIG["spatial_stats"]["permutations"])
grid_proj["lisa_I"] = lisa.Is
grid_proj["lisa_p"] = lisa.p_sim
grid_proj["lisa_q"] = lisa.q
cluster_map = {1: "HH", 2: "LH", 3: "LL", 4: "HL"}
grid_proj["lisa_cluster"] = grid_proj.apply(
    lambda r: cluster_map.get(r["lisa_q"], "NS") if r["lisa_p"] < sig else "NS", axis=1)
print(f"LISA clusters (p < {sig}):")
print(grid_proj["lisa_cluster"].value_counts().to_string())

cluster_colors = {"HH": "#1a9850", "LL": "#d73027", "HL": "#fdae61", "LH": "#abd9e9", "NS": "#e0e0e0"}
cluster_labels = {"HH": "Healthy cluster", "LL": "Unhealthy cluster", "HL": "Healthy outlier", "LH": "Unhealthy outlier", "NS": "Not significant"}
fig, ax = plt.subplots(1, 1, figsize=CONFIG["viz"]["figsize"])
for cl, color in cluster_colors.items():
    subset = grid_proj[grid_proj["lisa_cluster"] == cl]
    if len(subset) > 0: subset.plot(ax=ax, color=color, edgecolor="none", alpha=0.85)
if boundaries_loaded:
    city_proj.boundary.plot(ax=ax, color="white", linewidth=2, alpha=0.9)
    for _, row in city_proj.iterrows():
        c = row.geometry.centroid
        ax.annotate(row["name"], xy=(c.x, c.y), ha="center", va="center",
                    fontsize=10, fontweight="bold", color="white",
                    path_effects=[pe.withStroke(linewidth=3, foreground="black")])
legend_elements = [Patch(facecolor=c, label=f"{k}: {cluster_labels[k]}") for k, c in cluster_colors.items()]
ax.legend(handles=legend_elements, loc="upper right", fontsize=9, framealpha=0.9)
ax.set_title("LISA Clusters: Environmental Health Exposure\nKhordha and Cuttack Districts", fontsize=16, fontweight="bold", pad=15)
ax.set_axis_off()
plt.tight_layout()
plt.savefig("output/figures/04_lisa_clusters.png", dpi=CONFIG["viz"]["dpi"], bbox_inches="tight")
plt.show()

In [ ]:
w_binary = deepcopy(w)
w_binary.transform = "b"
gi = G_Local(y, w_binary, star=True, permutations=CONFIG["spatial_stats"]["permutations"])
grid_proj["gi_z"] = gi.Zs
grid_proj["gi_p"] = gi.p_sim
def classify_hotspot(z, p):
    if p > sig: return "Not Significant"
    if z > 0: return f"Hot Spot ({'99' if p < 0.01 else '95' if p < 0.05 else '90'}%)"
    else: return f"Cold Spot ({'99' if p < 0.01 else '95' if p < 0.05 else '90'}%)"
grid_proj["hotspot_class"] = grid_proj.apply(lambda r: classify_hotspot(r["gi_z"], r["gi_p"]), axis=1)
print(grid_proj["hotspot_class"].value_counts().to_string())

fig, ax = plt.subplots(1, 1, figsize=CONFIG["viz"]["figsize"])
vmin, vmax = grid_proj["gi_z"].quantile(0.02), grid_proj["gi_z"].quantile(0.98)
grid_proj.plot(column="gi_z", ax=ax, cmap="RdYlGn", legend=True, vmin=vmin, vmax=vmax,
    legend_kwds={"label": "Gi* Z-Score", "orientation": "horizontal", "shrink": 0.6}, edgecolor="none", alpha=0.85)
if boundaries_loaded:
    city_proj.boundary.plot(ax=ax, color="white", linewidth=2, alpha=0.9)
    for _, row in city_proj.iterrows():
        c = row.geometry.centroid
        ax.annotate(row["name"], xy=(c.x, c.y), ha="center", va="center",
                    fontsize=10, fontweight="bold", color="white",
                    path_effects=[pe.withStroke(linewidth=3, foreground="black")])
ax.set_title("Getis-Ord Gi* Hot/Cold Spots (EHEI)\nKhordha and Cuttack Districts", fontsize=16, fontweight="bold", pad=15)
ax.set_axis_off()
plt.tight_layout()
plt.savefig("output/figures/05_hotspots.png", dpi=CONFIG["viz"]["dpi"], bbox_inches="tight")
plt.show()

# **8. Temporal Change Analysis**

Comparing the 2016 and 2024 EHEI surfaces reveals not just where environmental health is poor today, but where it has deteriorated most rapidly over the eight-year period. The EHEI difference (2024 minus 2016) is computed per cell and classified into five change categories: major degradation, moderate degradation, stable, moderate improvement, and major improvement. Cells that were healthy in 2016 but degraded by 2024 represent active urbanization frontiers where intervention is most time-sensitive. Conversely, cells that improved may indicate successful planning measures or natural vegetation regeneration. The spatial distribution of change categories, overlaid with district boundaries and city annotations, provides planners with a visual prioritization framework.

In [ ]:
grid_proj["ehei_change"] = grid_proj["ehei_2024"] - grid_proj["ehei_2016"]
grid_proj["change_class"] = pd.cut(grid_proj["ehei_change"], bins=[-100,-15,-5,5,15,100],
    labels=["Major degradation","Moderate degradation","Stable","Moderate improvement","Major improvement"])
print(f"EHEI Change (2016 to 2024):")
print(f"  Mean change: {grid_proj['ehei_change'].mean():.1f}")
print(f"  Cells degraded: {(grid_proj['ehei_change'] < -5).sum()} ({(grid_proj['ehei_change'] < -5).mean()*100:.1f}%)")
print(f"\n{grid_proj['change_class'].value_counts().to_string()}")

fig, ax = plt.subplots(1, 1, figsize=CONFIG["viz"]["figsize"])
grid_proj.plot(column="ehei_change", ax=ax, cmap="RdYlGn", legend=True, vmin=-30, vmax=30,
    legend_kwds={"label": "EHEI Change (2016-2024)", "orientation": "horizontal", "shrink": 0.6}, edgecolor="none", alpha=0.85)
if boundaries_loaded:
    city_proj.boundary.plot(ax=ax, color="white", linewidth=2, alpha=0.9)
    for _, row in city_proj.iterrows():
        c = row.geometry.centroid
        ax.annotate(row["name"], xy=(c.x, c.y), ha="center", va="center",
                    fontsize=10, fontweight="bold", color="white",
                    path_effects=[pe.withStroke(linewidth=3, foreground="black")])
ax.set_title("Environmental Health Change (2016-2024)\nGreen=improved, Red=degraded", fontsize=16, fontweight="bold", pad=15)
ax.set_axis_off()
plt.tight_layout()
plt.savefig("output/figures/06_ehei_change.png", dpi=CONFIG["viz"]["dpi"], bbox_inches="tight")
plt.show()

# **9. Spatial Regression**

- OLS regression provides a baseline estimate of how LULC composition (vegetation, water, cropland fractions, and population) relates to EHEI, but standard OLS assumes independently distributed errors, an assumption almost certainly violated with spatially referenced data. The Lagrange Multiplier diagnostics built into PySAL's OLS implementation formally test for two types of spatial dependence: lag dependence (a cell's EHEI is partly a function of its neighbours' EHEI) and error dependence (unobserved spatially correlated factors drive residual variation). The Anselin (2005) decision tree guides model selection: if both LM tests are significant, the Robust LM variants break the tie.

- VIF diagnostics check for multicollinearity among the predictors. Built-up fraction is intentionally excluded because the four LULC fractions are compositional (they approximately sum to 1), so including all of them would create near-perfect collinearity. The built-up effect is implicitly captured as the reference category in the intercept.

- Spatial Lag (ML_Lag) and Spatial Error (ML_Error) models are estimated on a 5,000-cell subsample for computational feasibility, with Queen contiguity weights recomputed on the subsample. Model comparison by AIC determines the preferred specification, and coefficients from the winning model are exported for reporting.

In [ ]:
# Regress EHEI on LULC composition to understand which land use factors
# most strongly predict environmental health quality. OLS spatial diagnostics
# (LM tests) guide whether SLM or SEM is more appropriate.

from statsmodels.stats.outliers_influence import variance_inflation_factor

dv = "ehei_2024"
ivs = ["frac_vegetation_2024", "frac_water_2024",
       "frac_cropland_2024", "population_2024"]
# Dropped frac_builtup_2024: compositional collinearity (LULC fractions ~sum to 1).
# Builtup effect is implicitly captured as the reference category in the intercept.

y_reg = grid_proj[dv].values.reshape(-1, 1)
X_reg = grid_proj[ivs].values

ols = OLS(y_reg, X_reg, w=w, name_y=dv, name_x=ivs, spat_diag=True, moran=True)

t_arr = np.array(ols.t_stat)
ols_coefs = pd.DataFrame({
    "Variable": ["Constant"] + ivs,
    "Coef": ols.betas.flatten(), "SE": ols.std_err.flatten(),
    "t": t_arr[:, 0], "p": t_arr[:, 1],
})

print(f"OLS: R2={ols.r2:.4f}, Adj-R2={ols.ar2:.4f}, AIC={ols.aic:.1f}")
print(ols_coefs.round(4).to_string(index=False))

# VIF
vif_df = pd.DataFrame({
    "Variable": ivs,
    "VIF": [variance_inflation_factor(X_reg, i) for i in range(X_reg.shape[1])]
})
print(f"\nVIF:\n{vif_df.round(2).to_string(index=False)}")
# Note: VIF for frac_vegetation and frac_cropland may exceed 10 due to
# residual compositional dependence. Coefficient signs and significance
# remain stable across OLS and spatial models, so multicollinearity
# inflates SEs but does not distort inference materially.

# Spatial diagnostics
print(f"\nMoran's I (residuals): I={ols.moran_res[0]:.4f}, p={ols.moran_res[2]:.6f}")
print(f"LM-Lag:    {ols.lm_lag[0]:.4f} (p={ols.lm_lag[1]:.6f})")
print(f"LM-Error:  {ols.lm_error[0]:.4f} (p={ols.lm_error[1]:.6f})")
print(f"RLM-Lag:   {ols.rlm_lag[0]:.4f} (p={ols.rlm_lag[1]:.6f})")
print(f"RLM-Error: {ols.rlm_error[0]:.4f} (p={ols.rlm_error[1]:.6f})")

# Decision tree (Anselin 2005)
lm_lag_sig = ols.lm_lag[1] < 0.05
lm_err_sig = ols.lm_error[1] < 0.05
if lm_lag_sig and not lm_err_sig:
    recommendation = "Spatial Lag Model"
elif lm_err_sig and not lm_lag_sig:
    recommendation = "Spatial Error Model"
elif lm_lag_sig and lm_err_sig:
    recommendation = "SLM" if ols.rlm_lag[1] < ols.rlm_error[1] else "SEM"
else:
    recommendation = "OLS is adequate"
print(f"\n>>> Recommendation: {recommendation}")

ols_coefs.to_csv("output/ols_coefficients_ehei.csv", index=False)

In [ ]:
# Spatial models (subsampled for computational feasibility)
np.random.seed(42)
sample_size = min(5000, len(grid_proj))
s_idx = np.random.choice(len(grid_proj), sample_size, replace=False)
reg_sample = grid_proj.iloc[s_idx].reset_index(drop=True)

w_sample = Queen.from_dataframe(reg_sample, use_index=False)
w_sample.transform = "r"

y_s = reg_sample[dv].values.reshape(-1, 1)
X_s = reg_sample[ivs].values

print(f"Subsampled: {sample_size} / {len(grid_proj)} cells")
print(f"Islands in sample weights: {len(w_sample.islands)}\n")

# ── Spatial Lag Model ──
slm = ML_Lag(y_s, X_s, w=w_sample, name_y=dv, name_x=ivs)
z_slm = np.array(slm.z_stat)
slm_coefs = pd.DataFrame({
    "Variable": ["Constant"] + ivs + ["W_" + dv],
    "Coef": slm.betas.flatten(), "SE": slm.std_err.flatten(),
    "z": z_slm[:, 0], "p": z_slm[:, 1],
})
print(f"SLM: Pseudo-R2={slm.pr2:.4f}, AIC={slm.aic:.1f}, Rho={slm.betas[-1,0]:.4f}")
print(slm_coefs.round(4).to_string(index=False))

print("\n" + "-" * 60)

# ── Spatial Error Model ──
sem = ML_Error(y_s, X_s, w=w_sample, name_y=dv, name_x=ivs)

z_sem = np.array(sem.z_stat)
sem_coefs = pd.DataFrame({
    "Variable": ["Constant"] + ivs + ["Lambda"],
    "Coef": sem.betas.flatten(),
    "SE": sem.std_err.flatten(),
    "z": z_sem[:, 0],
    "p": z_sem[:, 1],
})
print(f"\nSEM: Pseudo-R2={sem.pr2:.4f}, AIC={sem.aic:.1f}, Lambda={sem.lam:.4f}")
print(sem_coefs.round(4).to_string(index=False))

# **10. Machine Learning**

Spatial regression reveals linear associations but cannot capture the non-linear interactions and threshold effects that characterize real urban environmental systems. This section applies two complementary ML approaches: Random Forest regression for prediction and feature importance, and K-Means clustering for spatial typology classification.


### **10.1. Random Forest + SHAP**

Random Forest regression fits an ensemble of decision trees (200 trees, max depth 15) that can model complex feature interactions without requiring explicit specification of interaction terms. Five-fold cross-validation provides an unbiased estimate of predictive accuracy, while the full-sample fit confirms the model has learned the signal rather than memorized the noise. The feature set includes the four LULC fractions from the regression model plus two temporal change variables (change_builtup, change_vegetation) that capture the dynamic dimension of environmental health.

SHAP (SHapley Additive exPlanations) values decompose each cell's prediction into feature-level contributions, answering not just "how well can we predict EHEI" but "which features matter most, and in which direction, for each individual cell." The beeswarm summary plot reveals whether features have linear or threshold effects, and whether their importance is symmetric or one-directional.

In [ ]:
ml_features = ivs + ["change_builtup", "change_vegetation"]
X_ml = grid_proj[ml_features].values
y_ml = grid_proj["ehei_2024"].values
rf = RandomForestRegressor(n_estimators=CONFIG["ml"]["rf_n_estimators"], max_depth=15,
    min_samples_leaf=10, random_state=SEED, n_jobs=-1)
cv_scores = cross_val_score(rf, X_ml, y_ml, cv=5, scoring="r2")
print(f"5-fold CV R2: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
rf.fit(X_ml, y_ml)
print(f"Full-sample R2: {r2_score(y_ml, rf.predict(X_ml)):.4f}")
print("\nComputing SHAP values...")
explainer = shap.TreeExplainer(rf)
X_shap = X_ml[np.random.choice(len(X_ml), min(2000, len(X_ml)), replace=False)]
shap_values = explainer.shap_values(X_shap)
fig, ax = plt.subplots(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap, feature_names=ml_features, show=False)
plt.tight_layout()
plt.savefig("output/figures/07_shap_summary.png", dpi=CONFIG["viz"]["dpi"], bbox_inches="tight")
plt.show()

### **10.2. K-Means Urban Environmental Typologies**

- Continuous index values are useful for analysis but difficult for planners to act on. Clustering the environmental feature space into discrete typologies groups cells with similar environmental health profiles into coherent spatial zones, where each typology implies a different policy response.

- The clustering pipeline includes three refinements beyond naive K-Means. First, latitude and longitude are added as features (with reduced weight) to encourage spatially contiguous clusters. Second, PCA reduces feature dimensionality and prevents any single variable from dominating the clustering. Third, a spatial majority filter (k=5 nearest neighbours) removes salt-and-pepper classification noise by reassigning isolated cells to their neighbourhood's dominant typology. Cluster labels are assigned automatically based on centroid characteristics (highest built-up = Dense Urban Core, highest vegetation = Vegetated Rural, etc.), ensuring labels remain interpretable even if cluster numbering changes across runs.

In [ ]:
from scipy.ndimage import median_filter
from scipy.spatial import cKDTree

cluster_features = ["frac_builtup_2024", "frac_vegetation_2024", "frac_water_2024",
                    "heat_norm_2024", "green_deficit_2024", "ehei_2024"]

# ── Step 1: Add spatial coordinates as features ──
grid_4326 = grid_proj.to_crs("EPSG:4326")
grid_proj["_lon"] = grid_4326.geometry.centroid.x.values
grid_proj["_lat"] = grid_4326.geometry.centroid.y.values
spatial_features = cluster_features + ["_lon", "_lat"]

# ── Step 2: Scale all features ──
scaler = StandardScaler()
X_scaled = scaler.fit_transform(grid_proj[spatial_features].values)

# ── Step 3: PCA to reduce feature dominance ──
pca_cluster = PCA(n_components=0.95, random_state=SEED)
X_pca = pca_cluster.fit_transform(X_scaled)
print(f"PCA: {X_scaled.shape[1]} features -> {X_pca.shape[1]} components "
      f"({pca_cluster.explained_variance_ratio_.sum():.3f} variance retained)")

# ── Step 4: K-Means on PCA components ──
for k in range(3, 8):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels, sample_size=min(5000, len(X_pca)))
    print(f"  k={k}: silhouette={sil:.3f}")

n_clusters = 4
km_final = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
raw_labels = km_final.fit_predict(X_pca)

# ── Step 5: Spatial majority filter to remove salt-and-pepper noise ──
centroids_xy = np.column_stack([
    grid_proj.geometry.centroid.x.values,
    grid_proj.geometry.centroid.y.values
])
tree = cKDTree(centroids_xy)
k_neighbors = 9
_, indices = tree.query(centroids_xy, k=k_neighbors)

smoothed_labels = np.empty_like(raw_labels)
for i in range(len(raw_labels)):
    neighbor_labels = raw_labels[indices[i]]
    values, counts = np.unique(neighbor_labels, return_counts=True)
    smoothed_labels[i] = values[counts.argmax()]

grid_proj["env_typology"] = smoothed_labels
pct_changed = (raw_labels != smoothed_labels).mean() * 100
print(f"\nSpatial smoothing: {pct_changed:.1f}% of cells reassigned to neighbor majority")

# ── Step 6: Label typologies from cluster profiles ──
centroids_inv = scaler.inverse_transform(
    pca_cluster.inverse_transform(km_final.cluster_centers_)
)
profile = pd.DataFrame(centroids_inv[:, :len(cluster_features)], columns=cluster_features)
print(f"\nCluster profiles:")
print(profile.round(3).to_string())

typology_names = {}
for idx, row in profile.iterrows():
    if row["frac_builtup_2024"] > 0.5:
        typology_names[idx] = "Dense Urban Core"
    elif row["heat_norm_2024"] > 0.4 and row["frac_builtup_2024"] > 0.3:
        typology_names[idx] = "Heat-Stressed Peri-Urban"
    elif row["frac_vegetation_2024"] > 0.45:
        typology_names[idx] = "Vegetated Rural"
    elif row["frac_water_2024"] > 0.08:
        typology_names[idx] = "Green-Blue Rural"
    else:
        typology_names[idx] = "Mixed Transitional"

print(f"\nAuto-assigned labels: {typology_names}")
grid_proj["env_typology_label"] = grid_proj["env_typology"].map(typology_names)
profile.to_csv("output/typology_profiles.csv", index=False)

# ── Step 7: Plot ──
fig, ax = plt.subplots(1, 1, figsize=CONFIG["viz"]["figsize"])
grid_proj.plot(column="env_typology_label", ax=ax, categorical=True, cmap="Set2",
    legend=True, legend_kwds={"title": "Environmental Typology", "loc": "upper right"},
    edgecolor="none", alpha=0.85)
if boundaries_loaded:
    city_proj.boundary.plot(ax=ax, color="white", linewidth=2, alpha=0.9)
    for _, row in city_proj.iterrows():
        c = row.geometry.centroid
        ax.annotate(row["name"], xy=(c.x, c.y), ha="center", va="center",
                    fontsize=10, fontweight="bold", color="white",
                    path_effects=[pe.withStroke(linewidth=3, foreground="black")])
ax.set_title("Urban Environmental Typologies (K-Means + Spatial Smoothing)\nKhordha and Cuttack Districts",
             fontsize=16, fontweight="bold", pad=15)
ax.set_axis_off()
plt.tight_layout()
plt.savefig("output/figures/08_env_typologies.png", dpi=CONFIG["viz"]["dpi"], bbox_inches="tight")
plt.show()

grid_proj.drop(columns=["_lon", "_lat"], inplace=True, errors="ignore")

# **11. Scenario-Based Green-Blue Intervention Modelling**

Understanding where environmental health is poor and why it is poor is necessary but not sufficient for planning. This section asks the forward-looking question: if targeted land cover interventions were implemented, how much environmental health improvement could realistically be achieved, and for how many people?

Three scenarios of increasing ambition are modelled. Each one modifies the 2024 land cover fractions, propagates those changes through all five exposure layers (heat stress, green deficit, blue deficit, air quality, industrial exposure), and recomputes the EHEI from scratch. The improvement is measured against the actual 2024 baseline on a fixed scale, so the results reflect absolute gains rather than relative re-ranking.

- **Scenario A (Minimal):** 5% conversion of barren and vacant land to green cover. A low-cost, low-disruption baseline representing what municipal bodies could achieve through existing urban greening schemes.
- **Scenario B (Moderate):** 15% land conversion plus restoration of riparian buffers along waterways. This reflects a coordinated intervention between urban planning and water resource management departments.
- **Scenario C (Ambitious):** 30% conversion, riparian restoration, and a connected green corridor network linking existing vegetation patches. This represents a long-term urban ecological restructuring aligned with national Smart Cities and AMRUT frameworks.

The progressive scaling from A to C tests whether incremental investment yields proportional or accelerating returns, a question central to infrastructure prioritization in resource-constrained settings.

In [ ]:
def simulate_scenario(grid_df, conversion_pct, riparian=False, corridors=False):
    sim = grid_df.copy()
    barren_available = sim["frac_barren_2024"].values * conversion_pct
    builtup_convertible = sim["frac_builtup_2024"].values * conversion_pct * 0.3
    sim["frac_vegetation_2024"] += barren_available + builtup_convertible
    sim["frac_barren_2024"] -= barren_available
    sim["frac_builtup_2024"] -= builtup_convertible
    if riparian:
        rp = sim["frac_water_2024"] > 0.05
        sim.loc[rp, "frac_vegetation_2024"] += 0.1
        sim.loc[rp, "frac_wetland_2024"] += 0.05
        sim.loc[rp, "frac_barren_2024"] -= 0.05
    if corridors:
        mg = (sim["frac_vegetation_2024"] > 0.1) & (sim["frac_vegetation_2024"] < 0.3)
        sim.loc[mg, "frac_vegetation_2024"] += 0.15
        sim.loc[mg, "frac_builtup_2024"] -= 0.05
    frac_cols = [c for c in sim.columns if c.startswith("frac_") and c.endswith("_2024")]
    sim[frac_cols] = sim[frac_cols].clip(lower=0)
    row_sums = sim[frac_cols].sum(axis=1)
    sim[frac_cols] = sim[frac_cols].div(row_sums, axis=0)
    sim["lst_sc"], sim["uhi_sc"], sim["heat_norm_2024"] = compute_heat_layer(sim, "2024")
    sim["green_pcap_sc"], sim["green_deficit_2024"] = compute_green_access(sim, "2024")
    sim["blue_deficit_2024"] = compute_blue_proximity(sim, "2024")
    sim["pm25_sc"], sim["air_norm_2024"] = compute_air_quality(sim, "2024")
    sim["industrial_norm_2024"] = compute_industrial_exposure(sim, "2024")
    exposure_cols = ["heat_norm_2024", "green_deficit_2024", "blue_deficit_2024", "air_norm_2024", "industrial_norm_2024"]
    X_exp = sim[exposure_cols].values
    ehei_raw = (X_exp * weights_2024).sum(axis=1)
    # Compute baseline raw exposure on the same scale
    baseline_exp = grid_df[exposure_cols].values
    baseline_raw = (baseline_exp * weights_2024).sum(axis=1)
    # Normalize both on the SAME scale (baseline min/max of raw exposure)
    raw_min = baseline_raw.min()
    raw_max = baseline_raw.max()
    ehei_scenario_norm = 100 * (1 - (ehei_raw - raw_min) / (raw_max - raw_min))
    return np.clip(ehei_scenario_norm, 0, 100)

scenario_results = {}
baseline_ehei = grid_proj["ehei_2024"].values
baseline_pop = grid_proj["population_2024"].values

# Recompute baseline EHEI on the same raw scale for fair comparison
baseline_exp_cols = ["heat_norm_2024", "green_deficit_2024", "blue_deficit_2024", "air_norm_2024", "industrial_norm_2024"]
baseline_exp_vals = grid_proj[baseline_exp_cols].values
baseline_raw = (baseline_exp_vals * weights_2024).sum(axis=1)
raw_min = baseline_raw.min()
raw_max = baseline_raw.max()
baseline_ehei_rescaled = 100 * (1 - (baseline_raw - raw_min) / (raw_max - raw_min))
baseline_ehei_rescaled = np.clip(baseline_ehei_rescaled, 0, 100)

for key, params in CONFIG["scenarios"].items():
    print(f"\nScenario {key} ({params['name']}):")
    ehei_new = simulate_scenario(grid_proj, params["conversion_pct"],
        riparian=params.get("riparian_buffer_m") is not None, corridors=params.get("corridors", False))
    improvement = ehei_new - baseline_ehei_rescaled
    was_poor = baseline_ehei_rescaled < 40
    now_better = ehei_new >= 40
    pop_improved = baseline_pop[was_poor & now_better].sum()
    scenario_results[key] = {"name": params["name"], "mean_improvement": improvement.mean(),
        "max_improvement": improvement.max(), "pop_improved": pop_improved,
        "pct_improved": (was_poor & now_better).mean() * 100, "ehei_new": ehei_new}
    print(f"  Mean EHEI improvement: +{improvement.mean():.1f}")
    print(f"  Population moved out of 'poor': {pop_improved:,.0f}")

scenario_df = pd.DataFrame([{"Scenario": f"{k}: {v['name']}", "Mean Improvement": v["mean_improvement"],
    "Max Improvement": v["max_improvement"], "Pop Improved": v["pop_improved"],
    "% Cells Improved": v["pct_improved"]} for k, v in scenario_results.items()])
print(f"\n{scenario_df.round(2).to_string(index=False)}")
scenario_df.to_csv("output/scenario_comparison.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for i, (key, result) in enumerate(scenario_results.items()):
    grid_proj[f"improvement_{key}"] = result["ehei_new"] - baseline_ehei
    grid_proj.plot(column=f"improvement_{key}", ax=axes[i], cmap="Greens", legend=True, vmin=0, vmax=20,
        legend_kwds={"label": "EHEI Improvement", "shrink": 0.5}, edgecolor="none")
    if boundaries_loaded:
        city_proj.boundary.plot(ax=axes[i], color="white", linewidth=1.5, alpha=0.8)
        for _, row in city_proj.iterrows():
            c = row.geometry.centroid
            axes[i].annotate(row["name"], xy=(c.x, c.y), ha="center", va="center",
                fontsize=8, fontweight="bold", color="white",
                path_effects=[pe.withStroke(linewidth=2.5, foreground="black")])
    axes[i].set_title(f"Scenario {key}: {result['name']}\nMean: +{result['mean_improvement']:.1f}", fontsize=13, fontweight="bold")
    axes[i].set_axis_off()
plt.suptitle("Green-Blue Infrastructure Intervention Scenarios", fontsize=17, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("output/figures/09_scenarios.png", dpi=CONFIG["viz"]["dpi"], bbox_inches="tight")
plt.show()

# **12. Interactive Dashboard**

The Folium-based dashboard provides togglable layers for every analytical output: EHEI for both years, temporal change, LISA clusters, Gi* hotspots, environmental typologies, and land cover fractions. Satellite basemap imagery allows users to ground-truth patterns against visible urban form, bridging the gap between abstract index values and the physical landscape they represent.

In [ ]:
import json as _json

print("Building interactive dashboard...")
m = folium.Map(location=CONFIG["study_area"]["center"], zoom_start=CONFIG["viz"]["zoom"],
               tiles=None, control_scale=True)

# ── Base tiles ──
folium.TileLayer("CartoDB positron", name="Light Basemap").add_to(m)
folium.TileLayer("CartoDB dark_matter", name="Dark Basemap").add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellite").add_to(m)

# ── Full grid in WGS84 (no subsampling) ──
grid_dash = grid_proj.to_crs("EPSG:4326").copy()

# ── Helper: build a choropleth layer from a column ──
def add_choropleth_layer(map_obj, gdf, column, cmap_colors, vmin, vmax,
                         caption, layer_name, show=False, tooltip_cols=None):
    cmap = LinearColormap(colors=cmap_colors, vmin=vmin, vmax=vmax, caption=caption)

    # Build style dict keyed by index for fast lookup
    vals = gdf[column].values
    def style_fn(feature):
        idx = feature["properties"]["_idx"]
        v = vals[idx]
        return {"fillColor": cmap(v), "color": "none", "fillOpacity": 0.7, "weight": 0}

    # Add index to properties for style lookup
    gdf_export = gdf.copy()
    gdf_export["_idx"] = range(len(gdf_export))

    # Build tooltip fields
    if tooltip_cols is None:
        tooltip_cols = [column]
    tooltip = folium.GeoJsonTooltip(fields=["_idx"] + tooltip_cols,
                                     aliases=["Cell"] + tooltip_cols,
                                     localize=True)

    layer = folium.FeatureGroup(name=layer_name, show=show)
    folium.GeoJson(
        gdf_export[["geometry", "_idx"] + tooltip_cols].to_json(),
        style_function=style_fn,
        tooltip=tooltip
    ).add_to(layer)
    layer.add_to(map_obj)
    if show:
        cmap.add_to(map_obj)
    return cmap

# ── Layer 1: EHEI 2024 ──
add_choropleth_layer(m, grid_dash, "ehei_2024",
    cmap_colors=["#d73027", "#fc8d59", "#fee08b", "#d9ef8b", "#1a9850"],
    vmin=0, vmax=100, caption="EHEI 2024 (0=Poor, 100=Excellent)",
    layer_name="EHEI 2024", show=True,
    tooltip_cols=["ehei_2024", "ehei_class_2024", "frac_builtup_2024",
                  "frac_vegetation_2024", "population_2024"])

# ── Layer 2: EHEI 2016 ──
add_choropleth_layer(m, grid_dash, "ehei_2016",
    cmap_colors=["#d73027", "#fc8d59", "#fee08b", "#d9ef8b", "#1a9850"],
    vmin=0, vmax=100, caption="EHEI 2016",
    layer_name="EHEI 2016", show=False,
    tooltip_cols=["ehei_2016"])

# ── Layer 3: EHEI Change ──
add_choropleth_layer(m, grid_dash, "ehei_change",
    cmap_colors=["#d73027", "#ffffff", "#1a9850"],
    vmin=-30, vmax=30, caption="EHEI Change (2016-2024)",
    layer_name="EHEI Change 2016-2024", show=False,
    tooltip_cols=["ehei_change"])

# ── Layer 4: LISA Clusters (categorical) ──
lisa_colors = {"HH": "#1a9850", "LL": "#d73027", "HL": "#fdae61", "LH": "#abd9e9", "NS": "#e0e0e0"}
lisa_labels = {"HH": "High-High", "LL": "Low-Low", "HL": "High-Low", "LH": "Low-High", "NS": "Not Significant"}

def lisa_style(feature):
    cluster = feature["properties"]["lisa_cluster"]
    c = lisa_colors.get(cluster, "#e0e0e0")
    opacity = 0.75 if cluster != "NS" else 0.1
    return {"fillColor": c, "color": "none", "fillOpacity": opacity, "weight": 0}

lisa_layer = folium.FeatureGroup(name="LISA Clusters", show=False)
lisa_gdf = grid_dash[["geometry", "lisa_cluster", "ehei_2024"]].copy()
folium.GeoJson(
    lisa_gdf.to_json(),
    style_function=lisa_style,
    tooltip=folium.GeoJsonTooltip(fields=["lisa_cluster", "ehei_2024"],
                                   aliases=["LISA Cluster", "EHEI 2024"])
).add_to(lisa_layer)
lisa_layer.add_to(m)

# ── Layer 5: Hotspot (Gi*) ──
add_choropleth_layer(m, grid_dash, "gi_z",
    cmap_colors=["#2166ac", "#67a9cf", "#f7f7f7", "#ef8a62", "#b2182b"],
    vmin=-4, vmax=4, caption="Getis-Ord Gi* (z-score)",
    layer_name="Hotspot Analysis (Gi*)", show=False,
    tooltip_cols=["gi_z", "hotspot_class"])

# ── Layer 6: Environmental Typologies (categorical) ──
typo_colors = {
    "Dense Urban Core": "#e41a1c",
    "Heat-Stressed Peri-Urban": "#ff7f00",
    "Mixed Transitional": "#ffff33",
    "Green-Blue Rural": "#377eb8",
    "Vegetated Rural": "#4daf4a"
}

def typo_style(feature):
    label = feature["properties"]["env_typology_label"]
    c = typo_colors.get(label, "#999999")
    return {"fillColor": c, "color": "none", "fillOpacity": 0.7, "weight": 0}

typo_layer = folium.FeatureGroup(name="Environmental Typologies", show=False)
typo_gdf = grid_dash[["geometry", "env_typology_label", "ehei_2024"]].copy()
folium.GeoJson(
    typo_gdf.to_json(),
    style_function=typo_style,
    tooltip=folium.GeoJsonTooltip(fields=["env_typology_label", "ehei_2024"],
                                   aliases=["Typology", "EHEI 2024"])
).add_to(typo_layer)
typo_layer.add_to(m)

# ── Layer 7: Built-up Fraction ──
add_choropleth_layer(m, grid_dash, "frac_builtup_2024",
    cmap_colors=["#ffffcc", "#fd8d3c", "#800026"],
    vmin=0, vmax=1, caption="Built-up Fraction 2024",
    layer_name="Built-up Fraction", show=False,
    tooltip_cols=["frac_builtup_2024"])

# ── Layer 8: Vegetation Fraction ──
add_choropleth_layer(m, grid_dash, "frac_vegetation_2024",
    cmap_colors=["#f7fcb1", "#78c679", "#005a32"],
    vmin=0, vmax=0.8, caption="Vegetation Fraction 2024",
    layer_name="Vegetation Fraction", show=False,
    tooltip_cols=["frac_vegetation_2024"])

# ── District Boundaries ──
if boundaries_loaded:
    dist_layer = folium.FeatureGroup(name="District Boundaries", show=True)
    for _, row in city_boundaries.iterrows():
        folium.GeoJson(row.geometry.__geo_interface__,
            style_function=lambda x: {"fillColor": "none", "color": "white", "weight": 2.5,
                                      "dashArray": "5,5", "fillOpacity": 0},
            tooltip=row["name"]).add_to(dist_layer)
    dist_layer.add_to(m)

# ── Layer Control ──
folium.LayerControl(collapsed=False).add_to(m)

# ── Title Panel ──
title_html = '''
<div style="position:fixed; top:10px; left:60px; z-index:9999;
    background:#ffffff; padding:12px 20px; border-radius:6px;
    box-shadow:0 1px 4px rgba(0,0,0,0.15); font-family:Georgia,serif;
    border-bottom:3px solid #2c5f2d;">
    <h4 style="margin:0; color:#1a1a1a; font-size:15px; font-weight:600;">
        Environmental Health Dashboard</h4>
    <p style="margin:3px 0 0 0; font-size:10px; color:#555; font-family:Arial,sans-serif;">
        Khordha & Cuttack Districts | EHEI | LISA | Gi* | Typologies | Scenarios</p>
</div>
'''
m.get_root().html.add_child(folium.Element(title_html))

# ── LISA Legend ──
lisa_legend_html = '''
<div style="position:fixed; bottom:30px; left:20px; z-index:9999;
    background:rgba(255,255,255,0.92); padding:12px 16px; border-radius:8px;
    box-shadow:0 2px 8px rgba(0,0,0,0.2); font-family:Arial,sans-serif; font-size:11px;">
    <b style="font-size:12px;">LISA Clusters</b><br>
    <span style="color:#1a9850;">&#9632;</span> HH: High-High (Good)<br>
    <span style="color:#d73027;">&#9632;</span> LL: Low-Low (Poor)<br>
    <span style="color:#fdae61;">&#9632;</span> HL: High-Low<br>
    <span style="color:#abd9e9;">&#9632;</span> LH: Low-High
</div>
'''
m.get_root().html.add_child(folium.Element(lisa_legend_html))

m.save("output/dashboard.html")
print("Dashboard saved: output/dashboard.html")
print(f"  Total cells rendered: {len(grid_dash)} (full grid, no subsampling)")
print(f"  Layers: EHEI 2024, EHEI 2016, EHEI Change, LISA, Gi* Hotspots,")
print(f"          Typologies, Built-up, Vegetation, Boundaries")
m

# **13. Summary and Export**

In [ ]:
n_total = len(grid_proj)
pop_total = grid_proj["population_2024"].sum()
poor_pop = grid_proj[grid_proj["ehei_class_2024"].isin(["Very Poor","Poor"])]["population_2024"].sum()
print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)
print(f"\n1. ENVIRONMENTAL HEALTH (EHEI 2024)")
print(f"   Mean EHEI: {grid_proj['ehei_2024'].mean():.1f} / 100")
print(f"   Population in poor/very poor: {poor_pop:,.0f} ({poor_pop/pop_total*100:.0f}%)")
print(f"\n2. TEMPORAL CHANGE (2016-2024)")
print(f"   Mean EHEI change: {grid_proj['ehei_change'].mean():.1f}")
print(f"   Built-up increase: +{grid_proj['change_builtup'].mean()*100:.1f} pp")
print(f"   Vegetation loss: {grid_proj['change_vegetation'].mean()*100:.1f} pp")
print(f"\n3. SPATIAL PATTERN")
print(f"   Moran's I = {mi.I:.4f} (p={mi.p_sim:.4f})")
print(f"   LL clusters: {(grid_proj['lisa_cluster']=='LL').sum()}, HH clusters: {(grid_proj['lisa_cluster']=='HH').sum()}")
print(f"\n4. REGRESSION")
print(f"   OLS R2={ols.r2:.4f}, SLM R2={slm.pr2:.4f}, SEM R2={sem.pr2:.4f}")
print(f"\n5. MACHINE LEARNING")
print(f"   RF CV-R2: {cv_scores.mean():.4f}")
print(f"   Environmental typologies: {n_clusters}")
print(f"\n6. SCENARIOS")
for k, v in scenario_results.items():
    print(f"   {k} ({v['name']}): +{v['mean_improvement']:.1f} EHEI, {v['pop_improved']:,.0f} people improved")
print(f"\n7. OUTPUTS")
print(f"   Figures: output/figures/ (9 PNG files)")
print(f"   Dashboard: output/dashboard.html")

In [ ]:
export_cols = ["cell_id","ehei_2024","ehei_class_2024","ehei_2016","ehei_change",
               "lisa_cluster","gi_z","hotspot_class","env_typology","geometry"]
grid_proj[export_cols].to_file("output/ehei_results.gpkg", driver="GPKG")
scenario_df.to_csv("output/scenario_comparison.csv", index=False)
profile.to_csv("output/typology_profiles.csv", index=False)
print("All outputs exported.")

import shutil
shutil.make_archive("ugbehei_output", "zip", "output")
try:
    from google.colab import files
    files.download("ugbehei_output.zip")
except ImportError:
    print("Not in Colab. Find ugbehei_output.zip in working directory.")